# Interactive Heterogeneity Analysis for Photoluminescence Images

This notebook provides an interactive interface for analyzing heterogeneity scores from H5 files containing photoluminescence (PL) images. 

## Features:
- Load and compare 2-4 H5 files
- Interactive parameter adjustment for all analysis settings
- Real-time visualization of processing pipeline
- Comprehensive results dashboard with sensitivity analysis

## Analysis Pipeline:
1. **Image Loading**: Extract PL images from H5 files
2. **Preprocessing**: Cropping and standardization
3. **Feature Extraction**: Entropy, standard deviation, and radial analysis
4. **Score Calculation**: Weighted combination of features
5. **Visualization**: Interactive plots of all intermediate steps

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Import Required Libraries

In [2]:
import h5py
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import matplotlib as mpl
from scipy.stats import entropy
from skimage.feature import graycomatrix, graycoprops
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Import functions from heterogeneity.py
from heterogeneity import (
    batch_processing, standardize, STD_PL, entropy_value_1, 
    radius_intensity, find_local_extrema_with_threshold, 
    extrema_variance, weight_avg, optimize_PL_image, my_heterogeneity_score
)

# Set matplotlib style
plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 2. Setup Interactive Widgets

Convert all hard-coded parameters into interactive widgets for real-time adjustment.

In [3]:
# Create interactive widgets for all parameters
print("Setting up interactive parameter widgets...")

# Image Processing Parameters
crop_widgets = {
    'y_start': widgets.IntSlider(value=300, min=0, max=1000, description='Crop Y Start:', style={'description_width': 'initial'}),
    'y_end': widgets.IntSlider(value=2450, min=1000, max=3000, description='Crop Y End:', style={'description_width': 'initial'}),
    'x_start': widgets.IntSlider(value=1000, min=0, max=2000, description='Crop X Start:', style={'description_width': 'initial'}),
    'x_end': widgets.IntSlider(value=3100, min=2000, max=4000, description='Crop X End:', style={'description_width': 'initial'})
}

# RGB handling option
rgb_handling_widgets = {
    'bayer_pattern': widgets.Dropdown(
        options=[
            ('None (No Bayer conversion)', None),
            ('BGGR', cv2.COLOR_BAYER_BG2RGB),
            ('GBRG', cv2.COLOR_BAYER_GB2RGB), 
            ('GRBG', cv2.COLOR_BAYER_GR2RGB),
            ('RGGB', cv2.COLOR_BAYER_RG2RGB)
        ],
        value=cv2.COLOR_BAYER_RG2RGB,
        description='Bayer Pattern:',
        style={'description_width': 'initial'}
    )
}

# Radial Analysis Parameters
radial_widgets = {
    'initial_radius': widgets.IntSlider(value=50, min=10, max=100, description='Initial Radius:', style={'description_width': 'initial'}),
    'radius_increment': widgets.IntSlider(value=50, min=10, max=100, description='Radius Increment:', style={'description_width': 'initial'}),
    'extrema_threshold': widgets.FloatSlider(value=0.02, min=0.001, max=0.1, step=0.001, description='Extrema Threshold:', style={'description_width': 'initial'})
}

# Polar Transform Parameters
polar_widgets = {
    'enable_polar': widgets.Checkbox(
        value=True,
        description='Enable Polar Transform Analysis',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    ),
    'polar_angles': widgets.IntSlider(value=360, min=180, max=720, description='Angular Resolution:', style={'description_width': 'initial'}),
    'center_offset_x': widgets.IntSlider(value=0, min=-100, max=100, description='Center Offset X:', style={'description_width': 'initial'}),
    'center_offset_y': widgets.IntSlider(value=0, min=-100, max=100, description='Center Offset Y:', style={'description_width': 'initial'})
}

# GLCM (Gray Level Co-occurrence Matrix) Parameters
glcm_widgets = {
    'enable_glcm': widgets.Checkbox(
        value=True,
        description='Enable GLCM Analysis',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    ),
    'distances': widgets.SelectMultiple(
        options=[1, 2, 3, 4, 5],
        value=[5],
        description='GLCM Distances:',
        style={'description_width': 'initial'}
    ),
    'angles': widgets.SelectMultiple(
        options=['0°', '45°', '90°', '135°'],
        value=['0°', '45°', '90°', '135°'],
        description='GLCM Angles:',
        style={'description_width': 'initial'}
    ),
    'levels': widgets.IntSlider(
        value=256, min=8, max=256, step=8,
        description='Gray Levels:',
        style={'description_width': 'initial'}
    ),
    # Individual GLCM property weights for composite score calculation
    'contrast_weight': widgets.FloatSlider(
        value=1.0, min=0, max=5, step=0.1,
        description='Contrast Weight:',
        style={'description_width': 'initial'}
    ),
    'dissimilarity_weight': widgets.FloatSlider(
        value=0.0, min=0, max=5, step=0.1,
        description='Dissimilarity Weight:',
        style={'description_width': 'initial'}
    ),
    'homogeneity_weight': widgets.FloatSlider(
        value=0.0, min=0, max=5, step=0.1,
        description='Homogeneity Weight:',
        style={'description_width': 'initial'}
    ),
    'energy_weight': widgets.FloatSlider(
        value=0.0, min=0, max=5, step=0.1,
        description='Energy Weight:',
        style={'description_width': 'initial'}
    ),
    'correlation_weight': widgets.FloatSlider(
        value=0.0, min=0, max=5, step=0.1,
        description='Correlation Weight:',
        style={'description_width': 'initial'}
    )
}

# Entropy Parameters
entropy_widgets = {
    'histogram_bins': widgets.IntSlider(value=101, min=50, max=200, description='Histogram Bins:', style={'description_width': 'initial'}),
    'use_custom_base': widgets.Checkbox(
        value=False,
        description='Use custom entropy base',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    ),
    'entropy_base': widgets.FloatSlider(value=2, min=2, max=10, description='Entropy Base:', style={'description_width': 'initial'})
}

# Normalization Factors
norm_widgets = {
    'entropy_norm': widgets.FloatText(value=1.0, description='Entropy Norm Factor:', style={'description_width': 'initial'}),
    'std_norm': widgets.FloatText(value=1.0, description='STD Norm Factor:', style={'description_width': 'initial'}),
    'extrema_norm': widgets.FloatText(value=1.0, description='Extrema Norm Factor:', style={'description_width': 'initial'}),
    'polar_concentric_norm': widgets.FloatText(value=1.0, description='Polar Concentric Norm:', style={'description_width': 'initial'}),
    'polar_radial_norm': widgets.FloatText(value=1.0, description='Polar Radial Norm:', style={'description_width': 'initial'}),
    'glcm_norm': widgets.FloatText(value=1.0, description='GLCM Norm Factor:', style={'description_width': 'initial'})
}

# Weight Coefficients
weight_widgets = {
    'entropy_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='Entropy Weight:', style={'description_width': 'initial'}),
    'std_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='STD Weight:', style={'description_width': 'initial'}),
    'extrema_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='Extrema Weight:', style={'description_width': 'initial'}),
    'polar_concentric_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='Polar Concentric Weight:', style={'description_width': 'initial'}),
    'polar_radial_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='Polar Radial Weight:', style={'description_width': 'initial'}),
    'glcm_weight': widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='GLCM Weight:', style={'description_width': 'initial'})
}

# File selection and output control
file_widgets = {
    'file_paths': widgets.Textarea(
        value='G:/My Drive/LPS/LPS-1/preprocessed/CBox/yrliu98_LPS-1_1_1_run1_spec_run.h5\n# Add more H5 file paths here (one per line)\n# Lines starting with # are ignored',
        placeholder='Enter H5 file paths...',
        description='H5 Files:',
        layout=widgets.Layout(width='100%', height='120px'),
        style={'description_width': 'initial'}
    ),
    'verbose': widgets.Checkbox(
        value=False,
        description='Verbose Output',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
}

print("✅ Interactive widgets created successfully!")

Setting up interactive parameter widgets...
✅ Interactive widgets created successfully!


## 3. File Reading and Image Loading Functions

In [4]:
def load_h5_image(file_path):
    """Load PL image from H5 file"""
    try:
        with h5py.File(file_path, 'r') as hf:
            # Check for 'adj_photo' or 'photo'
            if "measurement/spec_run/adj_photo" in hf:
                photo = hf["measurement/spec_run/adj_photo"][:]
                photo_type = "Adjusted Photo"
            elif "measurement/spec_run/photo" in hf:
                photo = hf["measurement/spec_run/photo"][:]
                photo_type = "Regular Photo"
            else:
                raise KeyError("Neither 'adj_photo' nor 'photo' found in the HDF5 file")

            # Detect image format (RGB or BW)
            original_shape = photo.shape
            if len(photo.shape) == 3:
                if photo.shape[2] == 3:
                    image_format = "RGB"
                elif photo.shape[2] == 4:
                    image_format = "RGBA"
                else:
                    image_format = f"Multi-channel ({photo.shape[2]} channels)"
            elif len(photo.shape) == 1:
                # Check if it might be flattened RGB
                total_pixels = photo.shape[0]
                if total_pixels % 3 == 0:
                    image_format = "Flattened RGB (1D array)"
                else:
                    image_format = "Flattened data (unknown format)"
            else:
                image_format = "Grayscale (BW)"

            # Ensure proper data type
            if photo.dtype != np.uint8:
                photo_normalized = ((photo - photo.min()) / (photo.max() - photo.min()) * 255).astype(np.uint8)
            else:
                photo_normalized = photo
                
            return photo_normalized, photo_type, image_format, original_shape, True
    except Exception as e:
        return None, str(e), None, None, False

def parse_file_paths(file_text):
    """Parse file paths from textarea input"""
    lines = file_text.strip().split('\n')
    paths = []
    for line in lines:
        line = line.strip()
        if line and not line.startswith('#'):
            paths.append(line)
    return paths

def load_all_files(file_paths, verbose=True):
    """Load all H5 files and return image data"""
    loaded_data = {}
    for i, path in enumerate(file_paths):
        if len(path.strip()) > 0:
            image, info, image_format, original_shape, success = load_h5_image(path.strip())
            if success:
                loaded_data[f"File_{i+1}"] = {
                    'path': path.strip(),
                    'image': image,
                    'info': info,
                    'format': image_format,
                    'original_shape': original_shape
                }
                if verbose:
                    print(f"✅ Loaded {path.split('/')[-1]} - {info} - {image_format} - Shape: {original_shape}")
            else:
                if verbose:
                    print(f"❌ Failed to load {path}: {info}")
    return loaded_data

print("✅ File loading functions defined!")

✅ File loading functions defined!


## 4. Modified Processing Functions with Widget Parameters

In [5]:
def batch_processing_interactive(raw_image_array, crop_params, rgb_params=None, verbose=True):
    """Modified batch processing with interactive crop parameters and Bayer conversion"""
    
    # Step 1: Handle Bayer pattern conversion if requested
    if rgb_params and rgb_params['bayer_pattern'] is not None:
        try:
            if verbose:
                print(f"    Converting Bayer pattern to RGB from shape {raw_image_array.shape}")
            # Ensure image is in proper format for Bayer conversion (single channel)
            if len(raw_image_array.shape) == 2:
                # Convert Bayer to RGB using the selected pattern
                raw_image_array = cv2.cvtColor(raw_image_array, rgb_params['bayer_pattern'])
                if verbose:
                    print(f"    Bayer converted to RGB: {raw_image_array.shape}")
            else:
                if verbose:
                    print(f"    ⚠️ Warning: Bayer conversion requires 2D input, got {len(raw_image_array.shape)}D")
        except Exception as e:
            if verbose:
                print(f"    ⚠️ Warning: Failed to convert Bayer pattern: {e}")
    
    # Step 2: Convert to grayscale if needed
    if len(raw_image_array.shape) == 3:
        if verbose:
            print("    Converting RGB to grayscale...")
        gray_image = cv2.cvtColor(raw_image_array, cv2.COLOR_RGB2HSV)[:, :, 2]  # Use V channel for brightness
    else:
        gray_image = raw_image_array
    
    # Step 3: Apply cropping
    y_start, y_end = crop_params['y_start'], crop_params['y_end'] 
    x_start, x_end = crop_params['x_start'], crop_params['x_end']
    
    if verbose:
        print(f"    Cropping from {gray_image.shape} to region [{y_start}:{y_end}, {x_start}:{x_end}]")
    image_cut_full = gray_image[y_start:y_end, x_start:x_end]
    if verbose:
        print(f"    Cropped to: {image_cut_full.shape}")
    
    return image_cut_full

def radius_intensity_interactive(image, radial_params):
    """Modified radius intensity with interactive parameters"""
    width, height = image.shape
    x = round(width / 2)
    y = round(height / 2)
    center_coordinates = (x, y)
    
    # Use interactive parameters
    radius = radial_params['initial_radius']
    radius_increment = radial_params['radius_increment']
    
    ring = 1
    intensity_rad_list = []
    
    # First circle
    mask_1 = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.circle(mask_1, center_coordinates, radius, 255, -1)
    pixel_255_locations = np.column_stack(np.where(mask_1 == 255))
    inside_circle = image[pixel_255_locations[:, 0], pixel_255_locations[:, 1]]
    average_value = np.average(inside_circle)
    intensity_rad_list.append(average_value)
    radius += radius_increment
    ring += 1
    
    # Continue until boundary
    while radius <= x and radius <= y:
        mask_2 = np.zeros(image.shape[:2], dtype=np.uint8)
        cv2.circle(mask_2, center_coordinates, radius, 255, -1)
        ring_between = cv2.subtract(mask_2, mask_1)
        pixel_255_locations = np.column_stack(np.where(ring_between == 255))
        inside_circle = image[pixel_255_locations[:, 0], pixel_255_locations[:, 1]]
        average_value = np.average(inside_circle)
        intensity_rad_list.append(average_value)
        radius += radius_increment
        ring += 1
        mask_1 = mask_2
    
    return np.array(intensity_rad_list)

def polar_transform_analysis(image, polar_params, verbose=True):
    """Perform polar transform analysis to measure concentric and radial patterns"""
    if not polar_params['enable_polar']:
        return None, None, None
    
    try:
        # Calculate center with optional offset
        center_x = image.shape[1] // 2 + polar_params['center_offset_x']
        center_y = image.shape[0] // 2 + polar_params['center_offset_y']
        center = (center_x, center_y)
        
        # Calculate maximum radius based on image bounds and center position
        max_radius = min(
            center_x, center_y,
            image.shape[1] - center_x, image.shape[0] - center_y
        )
        
        if max_radius <= 0:
            if verbose:
                print(f"    ⚠️ Warning: Invalid polar transform radius ({max_radius})")
            return None, None, None
        
        # Convert to polar coordinates
        angular_resolution = polar_params['polar_angles']
        polar_img = cv2.warpPolar(
            image, 
            (max_radius, angular_resolution), 
            center, 
            max_radius,
            cv2.WARP_FILL_OUTLIERS
        )
        
        # Measure variations
        # Variance along radius (axis=0) → concentric rings
        concentric_variance = np.std(polar_img, axis=0).mean()
        
        # Variance along angle (axis=1) → radial spokes  
        radial_variance = np.std(polar_img, axis=1).mean()
        
        if verbose:
            print(f"    Polar Transform: Concentric={concentric_variance:.2f}, Radial={radial_variance:.2f}")
        
        return concentric_variance, radial_variance, polar_img
        
    except Exception as e:
        if verbose:
            print(f"    ⚠️ Warning: Polar transform failed: {e}")
        return None, None, None

def glcm_analysis(image, glcm_params, verbose=True):
    """Perform Gray Level Co-occurrence Matrix (GLCM) texture analysis"""
    if not glcm_params['enable_glcm']:
        return None, None
    
    try:
        # Convert image to proper data type and scale for GLCM
        # GLCM requires integer values
        if image.max() <= 1:
            # Image is normalized (0-1), scale to gray levels
            img_scaled = (image * (glcm_params['levels'] - 1)).astype(np.uint8)
        else:
            # Image has values > 1, rescale to gray levels
            img_min, img_max = image.min(), image.max()
            img_scaled = ((image - img_min) / (img_max - img_min) * (glcm_params['levels'] - 1)).astype(np.uint8)
        
        # Convert angle strings to radians
        angle_map = {'0°': 0, '45°': np.pi/4, '90°': np.pi/2, '135°': 3*np.pi/4}
        angles = [angle_map[angle] for angle in glcm_params['angles']]
        
        # Calculate GLCM
        distances = list(glcm_params['distances'])
        
        # Compute GLCM matrix
        glcm = graycomatrix(
            img_scaled, 
            distances=distances, 
            angles=angles, 
            levels=glcm_params['levels'],
            symmetric=True, 
            normed=True
        )
        
        # Calculate all GLCM properties automatically
        all_properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
        glcm_features = {}
        for prop in all_properties:
            prop_values = graycoprops(glcm, prop)
            # Average across all distances and angles
            glcm_features[prop] = np.mean(prop_values)

        # Normalize contrast
        glcm_features['contrast'] = glcm_features['contrast'] / glcm_params['levels']
        glcm_features['dissimilarity'] = glcm_features['dissimilarity'] / glcm_params['levels']

        
        # Calculate composite GLCM score using individual property weights
        # Higher values indicate more texture/heterogeneity
        if len(glcm_features) > 0:
            # Get individual property weights
            prop_weights = glcm_params.get('property_weights', {
                'contrast_weight': 0.2,
                'dissimilarity_weight': 0.2,
                'homogeneity_weight': 0.2,
                'energy_weight': 0.2,
                'correlation_weight': 0.2
            })
            
            # Normalize each property to 0-1 range and apply individual weights
            weighted_sum = 0
            total_weights = 0
            
            for prop, value in glcm_features.items():
                # Get the weight for this property
                prop_weight_key = f'{prop}_weight'
                prop_weight = prop_weights.get(prop_weight_key, 0.2)
                
                # Normalize the property value
                if prop in ['contrast', 'dissimilarity']:
                    # Higher values = more heterogeneity
                    normalized_val = min(value / 100.0, 1.0)  # Scale contrast/dissimilarity
                elif prop == 'homogeneity':
                    # Lower values = more heterogeneity, so invert
                    normalized_val = 1.0 - value  
                elif prop == 'energy':
                    # Lower values = more heterogeneity, so invert
                    normalized_val = 1.0 - value
                elif prop == 'correlation':
                    # Use absolute value, higher = more structured
                    normalized_val = abs(value)
                else:
                    normalized_val = value
                
                # Add weighted contribution
                weighted_sum += normalized_val * prop_weight
                total_weights += prop_weight
            
            # Calculate weighted average
            composite_score = weighted_sum / total_weights if total_weights > 0 else 0
        else:
            composite_score = 0
            
        if verbose:
            print(f"    GLCM Analysis: {', '.join([f'{k}={v:.3f}' for k, v in glcm_features.items()])}")
            print(f"    GLCM Composite Score: {composite_score:.3f}")
        
        return composite_score, glcm_features
        
    except Exception as e:
        if verbose:
            print(f"    ⚠️ Warning: GLCM analysis failed: {e}")
        return None, None

def entropy_value_interactive(image, entropy_params):
    """Modified entropy calculation with interactive parameters - returns entropy and histogram data"""
    pixels = np.array(image)
    total_pixels = np.size(image)
    
    # Use interactive parameters
    bins = entropy_params['histogram_bins']
    use_custom_base = entropy_params.get('use_custom_base', True)
    base = entropy_params.get('entropy_base', 2) if use_custom_base else None
    
    hist, bin_edges = np.histogram(pixels, bins=bins, range=(0, 1), density=True)
    hist_normalized = hist / total_pixels
    
    # Calculate entropy with optional base parameter
    image_entropy = entropy(hist_normalized, base=base) / np.log(bins)
    
    # Return both entropy value and histogram data for visualization
    return image_entropy, hist_normalized, bin_edges

def weight_avg_interactive(entropy_val, STD_value, extrema_score, polar_concentric, polar_radial, glcm_score, norm_params, weight_params):
    """Modified weight average with interactive parameters including polar metrics and GLCM"""
    # Use interactive normalization factors
    normalize_entropy = entropy_val / norm_params['entropy_norm']
    normalize_STD = STD_value / norm_params['std_norm'] 
    normalize_extrema = extrema_score / norm_params['extrema_norm']
    
    # Normalize polar metrics (handle None values)
    normalize_polar_concentric = 0
    normalize_polar_radial = 0
    
    if polar_concentric is not None:
        normalize_polar_concentric = polar_concentric / norm_params['polar_concentric_norm']
    if polar_radial is not None:
        normalize_polar_radial = polar_radial / norm_params['polar_radial_norm']
    
    # Normalize GLCM score (handle None values)
    normalize_glcm = 0
    if glcm_score is not None:
        normalize_glcm = glcm_score / norm_params['glcm_norm']
    
    # Use weighted average mode: sum(feature*weight) / sum(weights)
    weighted_sum = (
        weight_params['entropy_weight'] * normalize_entropy + 
        weight_params['std_weight'] * normalize_STD + 
        weight_params['extrema_weight'] * normalize_extrema +
        weight_params['polar_concentric_weight'] * normalize_polar_concentric +
        weight_params['polar_radial_weight'] * normalize_polar_radial +
        weight_params['glcm_weight'] * normalize_glcm
    )
    
    total_weights = (
        weight_params['entropy_weight'] + 
        weight_params['std_weight'] + 
        weight_params['extrema_weight'] +
        weight_params['polar_concentric_weight'] +
        weight_params['polar_radial_weight'] +
        weight_params['glcm_weight']
    )
    
    final_score = weighted_sum / total_weights if total_weights > 0 else 0
    return final_score

def optimize_PL_image_interactive(raw_img_array, all_params, source_format=None):
    """Complete interactive processing pipeline with polar transform and GLCM"""
    # Extract parameter groups
    crop_params = all_params['crop']
    rgb_params = all_params.get('rgb', None)
    radial_params = all_params['radial'] 
    polar_params = all_params['polar']
    glcm_params = all_params['glcm']
    entropy_params = all_params['entropy']
    norm_params = all_params['norm']
    weight_params = all_params['weight']
    verbose = all_params.get('output', {}).get('verbose', True)
    
    # Processing pipeline
    cutted_img = batch_processing_interactive(raw_img_array, crop_params, rgb_params, verbose=verbose)
    standardize_img = standardize(cutted_img)
    entro_val, hist_normalized, bin_edges = entropy_value_interactive(standardize_img, entropy_params)
    STD_value = STD_PL(standardize_img)
    array_of_rings = radius_intensity_interactive(standardize_img, radial_params)
    location_of_extrema = find_local_extrema_with_threshold(array_of_rings, radial_params['extrema_threshold'])
    extrema_score = extrema_variance(array_of_rings, location_of_extrema)
    
    # Add polar transform analysis
    polar_concentric, polar_radial, polar_img = polar_transform_analysis(standardize_img, polar_params, verbose=verbose)
    
    # Add GLCM texture analysis
    glcm_score, glcm_features = glcm_analysis(standardize_img, glcm_params, verbose=verbose)
    
    final_score = weight_avg_interactive(entro_val, STD_value, extrema_score, polar_concentric, polar_radial, glcm_score, norm_params, weight_params)
    
    # Return intermediate results for visualization
    results = {
        'original': raw_img_array,
        'cropped': cutted_img,
        'standardized': standardize_img,
        'radial_profile': array_of_rings,
        'extrema_locations': location_of_extrema,
        'entropy': entro_val,
        'hist_normalized': hist_normalized,
        'bin_edges': bin_edges,
        'std': STD_value,
        'extrema_score': extrema_score,
        'polar_concentric': polar_concentric,
        'polar_radial': polar_radial,
        'polar_img': polar_img,
        'glcm_score': glcm_score,
        'glcm_features': glcm_features,
        'final_score': final_score,
        'source_format': source_format or "Unknown"
    }
    
    return results

print("✅ Interactive processing functions defined!")

✅ Interactive processing functions defined!


## 5. Visualization Functions

In [6]:
def plot_processing_pipeline(all_results, crop_params, radial_params, polar_params, norm_params, title="Processing Pipeline Comparison"):
    """Visualize the complete processing pipeline for multiple samples side by side"""
    n_files = len(all_results)
    if n_files == 0:
        return None
    
    # Create figure with appropriate size based on number of files, polar transform, and GLCM
    fig_width = max(20, 5 * n_files)
    polar_enabled = polar_params['enable_polar'] and any(results.get('polar_img') is not None for results in all_results.values())
    glcm_enabled = any(results.get('glcm_features') is not None for results in all_results.values())
    
    # Calculate number of rows: base (6) + polar (1) + GLCM (1)
    n_rows = 6
    if polar_enabled:
        n_rows += 1
    if glcm_enabled:
        n_rows += 1
    
    fig = plt.figure(figsize=(fig_width, 4 * n_rows))
    
    # Create grid layout: n_rows, n_files columns
    gs = GridSpec(n_rows, n_files, figure=fig, hspace=0.4, wspace=0.3)
    
    file_keys = list(all_results.keys())
    
    for col, (file_key, results) in enumerate(all_results.items()):
        row_offset = 0
        # Row 1: Original image with crop region
        ax1 = fig.add_subplot(gs[row_offset, col])
        ax1.imshow(results['original'], cmap='gray')
        ax1.set_title(f'{file_key}\nOriginal ({results.get("source_format", "Unknown")})')
        # Add crop rectangle
        rect = patches.Rectangle((crop_params['x_start'], crop_params['y_start']), 
                               crop_params['x_end']-crop_params['x_start'], 
                               crop_params['y_end']-crop_params['y_start'], 
                               linewidth=2, edgecolor='red', facecolor='none')
        ax1.add_patch(rect)
        ax1.set_xlabel('X pixels')
        ax1.set_ylabel('Y pixels')
        row_offset += 1
        
        # Row 2: Cropped image
        ax2 = fig.add_subplot(gs[row_offset, col])
        ax2.imshow(results['cropped'], cmap='gray')
        ax2.set_title(f'Cropped (→ BW)')
        ax2.set_xlabel('X pixels')
        ax2.set_ylabel('Y pixels')
        row_offset += 1
        
        # Row 3: Standardized image with rings overlay
        ax3 = fig.add_subplot(gs[row_offset, col])
        palette = plt.cm.viridis.with_extremes(over='m', under='r')
        pos = ax3.imshow(results['standardized'], cmap=palette, norm=mpl.colors.Normalize(vmin=0.05, vmax=0.95))
        fig.colorbar(pos, extend='both')
        row_offset += 1
        
        # Draw rings
        # center_x, center_y = results['standardized'].shape[1]//2, results['standardized'].shape[0]//2
        # radius = radial_params['initial_radius']
        # for i in range(len(results['radial_profile'])):
        #     circle = plt.Circle((center_x, center_y), radius, fill=False, color='white', linewidth=1, alpha=0.8)
        #     ax3.add_patch(circle)
        #     radius += radial_params['radius_increment']
        # ax3.set_title(f'Standardized + Rings')
        # ax3.set_xlabel('X pixels')
        # ax3.set_ylabel('Y pixels')
        
        # Row 4: Polar Transform (if enabled)
        if polar_enabled and results.get('polar_img') is not None:
            ax4 = fig.add_subplot(gs[3, col])
            palette = plt.cm.viridis.with_extremes(over='m', under='r')
            ax4.imshow(results['polar_img'], cmap=palette, norm=mpl.colors.Normalize(vmin=0.05, vmax=0.95), aspect='auto')
            ax4.set_title(f'Polar Transform')
            ax4.set_xlabel('Angle (degrees)')
            ax4.set_ylabel('Radius (pixels)')
            
            # Add polar metrics as text
            polar_text = f"Concentric: {results['polar_concentric']:.1f}\nRadial: {results['polar_radial']:.1f}"
            
            ax4.text(0.02, 0.98, polar_text, transform=ax4.transAxes, fontsize=9,
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            row_offset += 1
        
        # Add GLCM metrics bar plot (if enabled)
        if glcm_enabled and results.get('glcm_features') is not None:
            ax_glcm = fig.add_subplot(gs[row_offset, col])
            
            glcm_features = results['glcm_features']

            properties = list(glcm_features.keys())
            values = list(glcm_features.values())
            colors = ['red', 'orange', 'gold', 'lightblue', 'lightgreen'][:len(properties)]
            
            bars = ax_glcm.bar(properties, values, color=colors, alpha=0.7)
            ax_glcm.set_title(f'GLCM Texture Properties')
            ax_glcm.set_ylabel('Property Value')
            ax_glcm.tick_params(axis='x', rotation=45)
            ax_glcm.grid(True, alpha=0.3)
            ax_glcm.set_ylim(None, 1)
            
            # Add value labels on bars
            for bar, val in zip(bars, values):
                height = bar.get_height()
                height = height + height*0.01

                if height > ax_glcm.get_ylim()[1]:
                    ax_glcm.text(bar.get_x() + bar.get_width()/2., ax_glcm.get_ylim()[1] * 0.9,
                           f'{val:.3f}', ha='center', va='bottom', fontsize=8)
                else:
                    ax_glcm.text(bar.get_x() + bar.get_width()/2., height,
                            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
                
            
            # Add composite score as text
            composite_score = results.get('glcm_score', 0)
            ax_glcm.text(0.5, 0.98, f'Composite: {composite_score:.3f}', transform=ax_glcm.transAxes, 
                       fontsize=10, verticalalignment='top', horizontalalignment='center',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            row_offset += 1
        
        # Row 4/5/6: Normalized Histogram for Entropy Calculation
        ax_hist = fig.add_subplot(gs[row_offset, col])
        bin_centers = (results['bin_edges'][:-1] + results['bin_edges'][1:]) / 2
        ax_hist.bar(bin_centers, results['hist_normalized'], 
                width=np.diff(results['bin_edges'])[0], 
                alpha=0.7, color='skyblue', edgecolor='darkblue')
        ax_hist.set_title(f'Normalized Histogram')
        ax_hist.set_xlabel('Pixel Intensity (0-1)')
        ax_hist.set_ylabel('Normalized Frequency')
        ax_hist.grid(True, alpha=0.3)
        
        # Add entropy value as text
        ax_hist.text(0.02, 0.95, f'Entropy: {results["entropy"]:.4f}',
                transform=ax_hist.transAxes, fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
        row_offset += 1
        
        # Row 5/6: Radial intensity profile
        # ax_radial = fig.add_subplot(gs[4 + row_offset, col])
        # rings = np.arange(len(results['radial_profile']))
        # ax_radial.plot(rings, results['radial_profile'], 'b-', linewidth=2, label='Intensity')
        
        # Mark extrema
        # extrema_locs = results['extrema_locations']
        # extrema_vals = [results['radial_profile'][i] for i in extrema_locs]
        # ax_radial.scatter(extrema_locs, extrema_vals, color='red', s=40, zorder=5, label='Extrema')
        
        # ax_radial.set_title(f'Radial Profile')
        # ax_radial.set_xlabel('Ring Number')
        # ax_radial.set_ylabel('Intensity')
        # ax_radial.legend()
        # ax_radial.grid(True, alpha=0.3)
        
        # Row 6/7: Feature scores and summary
        ax_scores = fig.add_subplot(gs[row_offset, col])
        
        # Feature scores as horizontal bar chart - NORMALIZED values
        features = ['Entropy', 'STD', 'Extrema']
        # Calculate normalized scores for display
        norm_entropy = results['entropy'] / norm_params['entropy_norm']
        norm_std = results['std'] / norm_params['std_norm']
        norm_extrema = results['extrema_score'] / norm_params['extrema_norm']
        scores = [norm_entropy, norm_std, norm_extrema]
        colors = ['skyblue', 'lightgreen', 'orange']
        
        # Add polar metrics if available (normalized)
        if results.get('polar_concentric') is not None:
            features.extend(['Polar-C', 'Polar-R'])
            norm_polar_c = results['polar_concentric'] / norm_params['polar_concentric_norm']
            norm_polar_r = results['polar_radial'] / norm_params['polar_radial_norm']
            scores.extend([norm_polar_c, norm_polar_r])
            colors.extend(['purple', 'brown'])
        
        # Add GLCM metrics if available (normalized)
        if results.get('glcm_score') is not None:
            features.append('GLCM')
            norm_glcm = results['glcm_score'] / norm_params['glcm_norm']
            scores.append(norm_glcm)
            colors.append('magenta')
        
        features.append('Final')
        scores.append(results['final_score'])
        colors.append('red')
        
        bars = ax_scores.barh(features, scores, color=colors, alpha=0.7)
        ax_scores.set_title(f'Normalized Scores')
        ax_scores.set_xlabel('Normalized Score Value')
        ax_scores.set_xlim(None, 0.4)
        
        # Add value labels on bars
        for bar, score in zip(bars, scores):
            ax_scores.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                    f'{score:.3f}', ha='left', va='center', fontsize=9)
        
        # Add summary text below the bars
        polar_info = ""
        if results.get('polar_concentric') is not None:
            polar_info = f"Polar Transform: {polar_params['polar_angles']}° resolution\n"
        
        glcm_info = ""
        if results.get('glcm_score') is not None:
            glcm_properties = ', '.join(results.get('glcm_features', {}).keys()) if results.get('glcm_features') else 'N/A'
            glcm_info = f"GLCM Score: {results['glcm_score']:.3f} ({glcm_properties})\n"
            
        summary_text = f"""
Source: {results.get('source_format', 'Unknown')}
Shape: {results['original'].shape}
Rings: {len(results['radial_profile'])}
Extrema: {len(results['extrema_locations'])}
{polar_info}{glcm_info}Final Score: {results['final_score']:.4f}
        """
        ax_scores.text(0.02, -0.4, summary_text.strip(), transform=ax_scores.transAxes, fontsize=8,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.6))

    plt.suptitle(title, fontsize=16, y=0.98)
    plt.tight_layout()
    return fig

def plot_comparison_dashboard(all_results, file_data, norm_params):
    """Create a simplified comparison dashboard focusing on key metrics including polar transform"""
    n_files = len(all_results)
    if n_files == 0:
        return None
    
    # Check if polar transform data is available
    has_polar = any(results.get('polar_concentric') is not None for results in all_results.values())
    
    # Create a summary comparison plot
    fig_cols = 4 if has_polar else 3
    fig, axes = plt.subplots(2, fig_cols, figsize=(6 * fig_cols, 12))
    
    file_keys = list(all_results.keys())
    final_scores = [results['final_score'] for results in all_results.values()]
    # Use NORMALIZED scores for comparison
    entropy_scores = [results['entropy'] / norm_params['entropy_norm'] for results in all_results.values()]
    std_scores = [results['std'] / norm_params['std_norm'] for results in all_results.values()]
    extrema_scores = [results['extrema_score'] / norm_params['extrema_norm'] for results in all_results.values()]
    
    # Get polar scores if available (NORMALIZED)
    polar_concentric_scores = []
    polar_radial_scores = []
    if has_polar:
        for results in all_results.values():
            polar_c = results.get('polar_concentric', 0)
            polar_r = results.get('polar_radial', 0)
            # Normalize the polar scores
            norm_polar_c = polar_c / norm_params['polar_concentric_norm'] if polar_c != 0 else 0
            norm_polar_r = polar_r / norm_params['polar_radial_norm'] if polar_r != 0 else 0
            polar_concentric_scores.append(norm_polar_c)
            polar_radial_scores.append(norm_polar_r)
    
    # Get GLCM scores if available (NORMALIZED)
    has_glcm = any(results.get('glcm_score') is not None for results in all_results.values())
    glcm_scores = []
    if has_glcm:
        for results in all_results.values():
            glcm_score = results.get('glcm_score', 0)
            norm_glcm = glcm_score / norm_params['glcm_norm'] if glcm_score != 0 else 0
            glcm_scores.append(norm_glcm)
    
    # Score comparison bar chart - adjust width based on number of features
    x_pos = np.arange(len(file_keys))
    n_features = 3  # entropy, std, extrema (always present)
    if has_polar:
        n_features += 2  # polar concentric, polar radial
    if has_glcm:
        n_features += 1  # GLCM
    width = 0.8 / (n_features + 1)  # +1 for final score
    
    bar_pos = 0
    axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), entropy_scores, width, label='Entropy', alpha=0.7, color='skyblue')
    bar_pos += 1
    axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), std_scores, width, label='STD', alpha=0.7, color='lightgreen')
    bar_pos += 1
    axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), extrema_scores, width, label='Extrema', alpha=0.7, color='orange')
    bar_pos += 1
    
    if has_polar:
        axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), polar_concentric_scores, width, label='Polar-C', alpha=0.7, color='purple')
        bar_pos += 1
        axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), polar_radial_scores, width, label='Polar-R', alpha=0.7, color='brown')
        bar_pos += 1
    
    if has_glcm:
        axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), glcm_scores, width, label='GLCM', alpha=0.7, color='magenta')
        bar_pos += 1
    
    axes[0,0].bar(x_pos + width * (bar_pos - n_features/2), final_scores, width, label='Final', alpha=0.7, color='red')
    
    axes[0,0].set_xlabel('Files')
    axes[0,0].set_ylabel('Normalized Score Values')
    axes[0,0].set_title('Normalized Feature Score Comparison')
    axes[0,0].set_xticks(x_pos)
    axes[0,0].set_xticklabels(file_keys)
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Final scores only
    axes[0,1].bar(file_keys, final_scores, color='red', alpha=0.7)
    axes[0,1].set_title('Final Heterogeneity Scores')
    axes[0,1].set_ylabel('Final Score')
    axes[0,1].grid(True, alpha=0.3)
    
    # Add score values on bars
    for i, score in enumerate(final_scores):
        axes[0,1].text(i, score + 0.01, f'{score:.4f}', ha='center', va='bottom')
    
    # Histogram comparison overlay
    axes[0,2].set_title('Normalized Histograms Overlay')
    colors = plt.cm.tab10(np.linspace(0, 1, n_files))
    
    for i, (file_key, results) in enumerate(all_results.items()):
        bin_centers = (results['bin_edges'][:-1] + results['bin_edges'][1:]) / 2
        axes[0,2].plot(bin_centers, results['hist_normalized'], 
                      linewidth=2, label=f'{file_key} (E:{results["entropy"]:.3f})', 
                      color=colors[i])
    
    axes[0,2].set_xlabel('Pixel Intensity (0-1)')
    axes[0,2].set_ylabel('Normalized Frequency')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # Polar Transform Comparison (if available)
    if has_polar:
        axes[0,3].set_title('Normalized Polar Transform Metrics')
        x_pos = np.arange(len(file_keys))
        width = 0.35
        
        axes[0,3].bar(x_pos - width/2, polar_concentric_scores, width, label='Concentric Variance', 
                     alpha=0.7, color='purple')
        axes[0,3].bar(x_pos + width/2, polar_radial_scores, width, label='Radial Variance', 
                     alpha=0.7, color='brown')
        
        axes[0,3].set_xlabel('Files')
        axes[0,3].set_ylabel('Normalized Variance Score')
        axes[0,3].set_xticks(x_pos)
        axes[0,3].set_xticklabels(file_keys)
        axes[0,3].legend()
        axes[0,3].grid(True, alpha=0.3)
    
    # Radial profiles overlay
    axes[1,0].set_title('Radial Intensity Profiles Overlay')
    
    for i, (file_key, results) in enumerate(all_results.items()):
        rings = np.arange(len(results['radial_profile']))
        axes[1,0].plot(rings, results['radial_profile'], 
                      linewidth=2, label=file_key, color=colors[i])
    
    axes[1,0].set_xlabel('Ring Number')
    axes[1,0].set_ylabel('Average Intensity')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Normalized Entropy vs Final Score scatter plot
    axes[1,1].scatter(entropy_scores, final_scores, c=colors[:n_files], s=100, alpha=0.7)
    axes[1,1].set_xlabel('Normalized Entropy Score')
    axes[1,1].set_ylabel('Final Score')
    axes[1,1].set_title('Normalized Entropy vs Final Score')
    axes[1,1].grid(True, alpha=0.3)
    
    # Add file labels to scatter points
    for i, file_key in enumerate(file_keys):
        axes[1,1].annotate(file_key, (entropy_scores[i], final_scores[i]), 
                          xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    # Statistics summary with format and polar information
    axes[1,2].axis('off')
    
    # Collect format information
    format_info = []
    for file_key, results in all_results.items():
        source_format = results.get('source_format', 'Unknown')
        format_info.append(f"  • {file_key}: {source_format}")
    
    # Add polar statistics if available
    polar_stats = ""
    if has_polar:
        # Get angular resolution safely
        angular_res = 'N/A'
        first_result = list(all_results.values())[0]
        if first_result.get('polar_img') is not None:
            polar_img = first_result['polar_img']
            if hasattr(polar_img, 'shape') and len(polar_img.shape) > 1:
                angular_res = polar_img.shape[1]
        
        polar_stats = f"""
Normalized Polar Transform:
• Concentric Range: {min(polar_concentric_scores):.3f} - {max(polar_concentric_scores):.3f}
• Radial Range: {min(polar_radial_scores):.3f} - {max(polar_radial_scores):.3f}
• Angular Resolution: {angular_res}°
"""
    
    # Add GLCM statistics if available
    glcm_stats = ""
    if has_glcm:
        glcm_stats = f"""
Normalized GLCM Texture:
• GLCM Range: {min(glcm_scores):.3f} - {max(glcm_scores):.3f}
• GLCM Average: {np.mean(glcm_scores):.3f}
"""
    
    stats_text = f"""
Summary Statistics:

Files Processed: {n_files}

Source Formats:
{chr(10).join(format_info)}

Final Scores:
• Range: {min(final_scores):.4f} - {max(final_scores):.4f}
• Mean: {np.mean(final_scores):.4f}
• Std Dev: {np.std(final_scores):.4f}

Normalized Feature Averages:
• Entropy: {np.mean(entropy_scores):.4f}
• STD: {np.mean(std_scores):.4f}  
• Extrema: {np.mean(extrema_scores):.4f}
{polar_stats}{glcm_stats}
    """
    
    axes[1,2].text(0.05, 0.95, stats_text, transform=axes[1,2].transAxes, fontsize=10,
                  verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    # Polar vs Other Metrics Correlation (if available and space permits)
    if has_polar and fig_cols >= 4:
        axes[1,3].set_title('Polar Metrics Correlation')
        axes[1,3].scatter(polar_concentric_scores, polar_radial_scores, c=colors[:n_files], s=100, alpha=0.7)
        axes[1,3].set_xlabel('Concentric Variance')
        axes[1,3].set_ylabel('Radial Variance')
        axes[1,3].grid(True, alpha=0.3)
        
        # Add file labels
        for i, file_key in enumerate(file_keys):
            axes[1,3].annotate(file_key, (polar_concentric_scores[i], polar_radial_scores[i]), 
                              xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    title_parts = ['Enhanced Comparison Dashboard']
    if has_polar:
        title_parts.append('Polar Transform')
    if has_glcm:
        title_parts.append('GLCM Texture')
    
    plt.suptitle(' + '.join(title_parts), fontsize=14)
    plt.tight_layout()
    return fig

print("✅ Visualization functions defined!")

✅ Visualization functions defined!


## 6. Main Interactive Interface

This section creates the main interactive dashboard where you can:
- Load H5 files 
- Adjust all processing parameters in real-time
- Visualize the complete analysis pipeline
- Compare results across multiple files

In [7]:
# Create organized widget layout
print("Setting up main interactive interface...")

# File Selection Section
file_section = widgets.VBox([
    widgets.HTML("<h3>📁 File Selection</h3>"),
    file_widgets['file_paths'],
    widgets.HTML("<i>Enter full paths to your H5 files, one per line. Lines starting with # are ignored.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>📊 Output Control:</h4>"),
    file_widgets['verbose'],
    widgets.HTML("<i>Show detailed processing information (format detection, cropping, polar transforms, GLCM analysis, etc.)</i>")
])

# Bayer Conversion Section
bayer_section = widgets.VBox([
    widgets.HTML("<h3>🎨 Bayer Pattern Conversion</h3>"),
    
    # Bayer pattern conversion
    rgb_handling_widgets['bayer_pattern'],
    widgets.HTML("<i>Select 'None' to skip Bayer conversion, or choose the appropriate Bayer pattern (BGGR, GBRG, GRBG, RGGB) for your raw sensor data.</i>"),
    
    widgets.HTML("<hr>"),
    widgets.HTML("<i><b>Processing Order:</b> Bayer → RGB → Grayscale → Crop</i>")
])

# Image Processing Parameters Section  
crop_section = widgets.VBox([
    widgets.HTML("<h3>✂️ Image Cropping Parameters</h3>"),
    widgets.HBox([crop_widgets['y_start'], crop_widgets['y_end']]),
    widgets.HBox([crop_widgets['x_start'], crop_widgets['x_end']]),
    widgets.HTML("<i>Note: Cropping is applied after Bayer conversion (if enabled).</i>")
])

# Radial Analysis Parameters Section
radial_section = widgets.VBox([
    widgets.HTML("<h3>🎯 Radial Analysis Parameters</h3>"),
    radial_widgets['initial_radius'],
    radial_widgets['radius_increment'], 
    radial_widgets['extrema_threshold']
])

# Polar Transform Parameters Section
polar_section = widgets.VBox([
    widgets.HTML("<h3>🌀 Polar Transform Analysis</h3>"),
    polar_widgets['enable_polar'],
    widgets.HTML("<i>Enable polar coordinate transformation to measure concentric and radial pattern variations.</i>"),
    widgets.HTML("<hr>"),
    polar_widgets['polar_angles'],
    widgets.HTML("<i>Angular resolution for polar transform (180-720 degrees). Higher values provide more detail.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>Center Adjustment:</h4>"),
    widgets.HBox([polar_widgets['center_offset_x'], polar_widgets['center_offset_y']]),
    widgets.HTML("<i>Fine-tune the center point for polar transformation (±100 pixels from image center).</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<i><b>Metrics:</b><br>• <b>Concentric Variance:</b> Measures ring-like patterns<br>• <b>Radial Variance:</b> Measures spoke-like patterns</i>")
])

# GLCM Parameters Section
glcm_section = widgets.VBox([
    widgets.HTML("<h3>🔬 GLCM (Gray Level Co-occurrence Matrix) Analysis</h3>"),
    glcm_widgets['enable_glcm'],
    widgets.HTML("<i>Enable texture analysis using Gray Level Co-occurrence Matrix to measure spatial relationships between pixels.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>Distance and Direction Settings:</h4>"),
    glcm_widgets['distances'],
    widgets.HTML("<i>Pixel distances for co-occurrence analysis (1-5 pixels). Multiple values analyze different scales.</i>"),
    glcm_widgets['angles'],
    widgets.HTML("<i>Directional angles for texture analysis. Select multiple angles to capture different orientations.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>Analysis Parameters:</h4>"),
    glcm_widgets['levels'],
    widgets.HTML("<i>Number of gray levels for GLCM computation (8-256). Higher values = more detail, slower computation.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>🎛️ GLCM Property Weights:</h4>"),
    widgets.HTML("<i>Adjust the relative importance of each texture property in the composite GLCM score:</i>"),
    glcm_widgets['contrast_weight'],
    widgets.HTML("<i>Contrast: Measures local intensity variations between neighboring pixels</i>"),
    glcm_widgets['dissimilarity_weight'],
    widgets.HTML("<i>Dissimilarity: Measures the amount of local variation in the image</i>"),
    glcm_widgets['homogeneity_weight'],
    widgets.HTML("<i>Homogeneity: Measures texture uniformity (inverted for heterogeneity)</i>"),
    glcm_widgets['energy_weight'],
    widgets.HTML("<i>Energy: Measures texture orderliness (inverted for heterogeneity)</i>"),
    glcm_widgets['correlation_weight'],
    widgets.HTML("<i>Correlation: Measures linear dependencies between pixel values</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<i><b>Weighted Average:</b> Higher weights give more importance to specific texture properties.<br><b>All 5 properties are calculated automatically</b> and combined using weighted averaging.</i>")
])

# Entropy Parameters Section
entropy_section = widgets.VBox([
    widgets.HTML("<h3>📊 Entropy Parameters</h3>"),
    entropy_widgets['histogram_bins'],
    widgets.HTML("<hr>"),
    entropy_widgets['use_custom_base'],
    widgets.HTML("<i>When disabled, uses natural logarithm (base e) for entropy calculation.</i>"),
    entropy_widgets['entropy_base'],
    widgets.HTML("<i>Specify custom base for entropy calculation (typically 2 for bits).</i>")
])

# Normalization Factors Section
norm_section = widgets.VBox([
    widgets.HTML("<h3>⚖️ Normalization Factors</h3>"),
    widgets.HTML("<h4>Traditional Metrics:</h4>"),
    norm_widgets['entropy_norm'],
    norm_widgets['std_norm'],
    norm_widgets['extrema_norm'],
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>Polar Transform Metrics:</h4>"),
    norm_widgets['polar_concentric_norm'],
    norm_widgets['polar_radial_norm'],
    widgets.HTML("<i>Normalization factors for polar transform variance measures.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>GLCM Texture Metrics:</h4>"),
    norm_widgets['glcm_norm'],
    widgets.HTML("<i>Normalization factor for GLCM composite texture score.</i>")
])

# Weight Coefficients Section
weight_section = widgets.VBox([
    widgets.HTML("<h3>⚡ Feature Weights</h3>"),
    widgets.HTML("<h4>Traditional Features:</h4>"),
    weight_widgets['entropy_weight'],
    weight_widgets['std_weight'], 
    weight_widgets['extrema_weight'],
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>Polar Transform Features:</h4>"),
    weight_widgets['polar_concentric_weight'],
    weight_widgets['polar_radial_weight'],
    widgets.HTML("<hr>"),
    widgets.HTML("<h4>GLCM Texture Features:</h4>"),
    weight_widgets['glcm_weight'],
    widgets.HTML("<i>Weight for GLCM texture analysis composite score.</i>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<i><b>Weighted Average:</b> Higher weights give more importance to specific features. No normalization required - uses weighted averaging formula.</i>")
])



# Add a callback for Bayer pattern selection to provide feedback
def on_bayer_pattern_change(change):
    """Provide feedback when Bayer pattern is changed"""
    if change['new'] is not None:
        pattern_name = [k for k, v in rgb_handling_widgets['bayer_pattern'].options if v == change['new']][0]
        print(f"🎨 Bayer conversion enabled: {pattern_name}")
    else:
        print("🎨 Bayer conversion disabled")

rgb_handling_widgets['bayer_pattern'].observe(on_bayer_pattern_change, names='value')

# Add callback for entropy base checkbox
def on_entropy_base_toggle(change):
    """Enable/disable entropy base slider when checkbox is toggled"""
    entropy_widgets['entropy_base'].disabled = not change['new']
    if change['new']:
        print("📊 Custom entropy base enabled")
    else:
        print("📊 Using natural logarithm (base e) for entropy calculation")

entropy_widgets['use_custom_base'].observe(on_entropy_base_toggle, names='value')

# Add callback for polar transform enable/disable
def on_polar_enable_toggle(change):
    """Provide feedback when polar transform is toggled"""
    if change['new']:
        print("🌀 Polar transform analysis enabled")
    else:
        print("🌀 Polar transform analysis disabled")

polar_widgets['enable_polar'].observe(on_polar_enable_toggle, names='value')

# Add callback for GLCM enable/disable
def on_glcm_enable_toggle(change):
    """Provide feedback when GLCM analysis is toggled"""
    if change['new']:
        print("🔬 GLCM texture analysis enabled")
    else:
        print("🔬 GLCM texture analysis disabled")

glcm_widgets['enable_glcm'].observe(on_glcm_enable_toggle, names='value')

# Control buttons
analyze_button = widgets.Button(
    description='🔬 Analyze Files',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

reset_button = widgets.Button(
    description='🔄 Reset Parameters', 
    button_style='warning',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area for results
output_area = widgets.Output()

print("✅ Main interface widgets created!")

Setting up main interactive interface...
✅ Main interface widgets created!


In [8]:
# Note: Dashboard setup moved to cell below for better organization
print("✅ Dashboard components prepared!")

✅ Dashboard components prepared!


## 7. Interactive Dashboard

Run the cell below to display the interactive dashboard. You can:

1. **Load Files**: Enter H5 file paths in the text area (one per line)
2. **Adjust Parameters**: Use the sliders and input fields to modify analysis parameters
3. **Run Analysis**: Click "🔬 Analyze Files" to process your data with current settings
4. **Reset**: Click "🔄 Reset Parameters" to restore default values

### Parameter Guide:
- **Cropping**: Define the region of interest in the original image
- **Radial Analysis**: Control ring spacing and extrema detection sensitivity
- **Polar Transform**: Analyze concentric and radial patterns using polar coordinates  
- **GLCM Texture**: Gray Level Co-occurrence Matrix for spatial texture analysis
- **Entropy**: Adjust histogram binning and entropy calculation
- **Normalization**: Calibration factors (usually kept as defaults)
- **Weights**: Relative importance of each feature in final score (should sum to 1.0)

In [9]:
# Display the complete interactive dashboard
dashboard = widgets.VBox([
    widgets.HTML("<h2>🔬 Interactive Heterogeneity Analysis Dashboard</h2>"),
    
    # File selection at the top
    file_section,
    
    widgets.HTML("<hr>"),
    
    # Parameter sections in tabs for better organization
    widgets.Tab(children=[
        bayer_section,
        crop_section,
        radial_section,
        polar_section,
        glcm_section,
        entropy_section,
        norm_section,
        weight_section
    ]),
    
    widgets.HTML("<hr>"),
    
    # Control buttons
    widgets.HBox([
        analyze_button,
        reset_button
    ], layout=widgets.Layout(justify_content='center')),
    
    widgets.HTML("<hr>"),
    
    # Output area
    output_area
])

# Set tab titles
dashboard.children[3].set_title(0, 'Bayer Conversion')
dashboard.children[3].set_title(1, 'Image Cropping')
dashboard.children[3].set_title(2, 'Radial Analysis')
dashboard.children[3].set_title(3, 'Polar Transform')
dashboard.children[3].set_title(4, 'GLCM Texture')
dashboard.children[3].set_title(5, 'Entropy Parameters')
dashboard.children[3].set_title(6, 'Normalization')
dashboard.children[3].set_title(7, 'Feature Weights')

# Display the dashboard
display(dashboard)

print("\n🎉 Interactive Dashboard Ready!")
print("👆 Enter your H5 file paths above and click 'Analyze Files' to begin!")
print("🌀 NEW: Polar Transform analysis now available for measuring concentric and radial patterns!")
print("🔬 NEW: GLCM (Gray Level Co-occurrence Matrix) texture analysis now available!")


🎉 Interactive Dashboard Ready!
👆 Enter your H5 file paths above and click 'Analyze Files' to begin!
🌀 NEW: Polar Transform analysis now available for measuring concentric and radial patterns!
🔬 NEW: GLCM (Gray Level Co-occurrence Matrix) texture analysis now available!


In [10]:
# Analysis and Reset Button Callbacks
def collect_all_parameters():
    """Collect all current parameter values from widgets"""
    return {
        'crop': {
            'y_start': crop_widgets['y_start'].value,
            'y_end': crop_widgets['y_end'].value,
            'x_start': crop_widgets['x_start'].value,
            'x_end': crop_widgets['x_end'].value
        },
        'rgb': {
            'bayer_pattern': rgb_handling_widgets['bayer_pattern'].value
        },
        'radial': {
            'initial_radius': radial_widgets['initial_radius'].value,
            'radius_increment': radial_widgets['radius_increment'].value,
            'extrema_threshold': radial_widgets['extrema_threshold'].value
        },
        'polar': {
            'enable_polar': polar_widgets['enable_polar'].value,
            'polar_angles': polar_widgets['polar_angles'].value,
            'center_offset_x': polar_widgets['center_offset_x'].value,
            'center_offset_y': polar_widgets['center_offset_y'].value
        },
        'glcm': {
            'enable_glcm': glcm_widgets['enable_glcm'].value,
            'distances': glcm_widgets['distances'].value,
            'angles': glcm_widgets['angles'].value,
            'levels': glcm_widgets['levels'].value,
            'property_weights': {
                'contrast_weight': glcm_widgets['contrast_weight'].value,
                'dissimilarity_weight': glcm_widgets['dissimilarity_weight'].value,
                'homogeneity_weight': glcm_widgets['homogeneity_weight'].value,
                'energy_weight': glcm_widgets['energy_weight'].value,
                'correlation_weight': glcm_widgets['correlation_weight'].value
            }
        },
        'entropy': {
            'histogram_bins': entropy_widgets['histogram_bins'].value,
            'use_custom_base': entropy_widgets['use_custom_base'].value,
            'entropy_base': entropy_widgets['entropy_base'].value
        },
        'norm': {
            'entropy_norm': norm_widgets['entropy_norm'].value,
            'std_norm': norm_widgets['std_norm'].value,
            'extrema_norm': norm_widgets['extrema_norm'].value,
            'polar_concentric_norm': norm_widgets['polar_concentric_norm'].value,
            'polar_radial_norm': norm_widgets['polar_radial_norm'].value,
            'glcm_norm': norm_widgets['glcm_norm'].value
        },
        'weight': {
            'entropy_weight': weight_widgets['entropy_weight'].value,
            'std_weight': weight_widgets['std_weight'].value,
            'extrema_weight': weight_widgets['extrema_weight'].value,
            'polar_concentric_weight': weight_widgets['polar_concentric_weight'].value,
            'polar_radial_weight': weight_widgets['polar_radial_weight'].value,
            'glcm_weight': weight_widgets['glcm_weight'].value
        },
        'output': {
            'verbose': file_widgets['verbose'].value
        }
    }

def run_analysis(button):
    """Main analysis function triggered by the Analyze button"""
    with output_area:
        clear_output(wait=True)
        
        try:
            # Collect current parameters first to get verbose setting
            all_params = collect_all_parameters()
            verbose = all_params.get('output', {}).get('verbose', True)
            
            if verbose:
                print("🔬 Starting heterogeneity analysis...")
            
            # Parse file paths
            file_paths = parse_file_paths(file_widgets['file_paths'].value)
            if not file_paths:
                print("❌ No valid file paths provided!")
                return
            
            if verbose:
                print(f"📁 Found {len(file_paths)} file(s) to process")
            
            # Load all files
            if verbose:
                print("📂 Loading H5 files...")
            loaded_data = load_all_files(file_paths, verbose=verbose)
            
            if not loaded_data:
                print("❌ No files were successfully loaded!")
                return
            
            # Show parameter summary
            if verbose:
                print(f"\n⚙️ Analysis Parameters:")
                if all_params['rgb']['bayer_pattern'] is not None:
                    pattern_name = [k for k, v in rgb_handling_widgets['bayer_pattern'].options if v == all_params['rgb']['bayer_pattern']][0]
                    print(f"  • Bayer Conversion: Enabled ({pattern_name})")
                else:
                    print(f"  • Bayer Conversion: Disabled")
                print(f"  • Crop Region: [{all_params['crop']['y_start']}:{all_params['crop']['y_end']}, {all_params['crop']['x_start']}:{all_params['crop']['x_end']}]")
                print(f"  • Radial Analysis: {all_params['radial']['initial_radius']}px start, {all_params['radial']['radius_increment']}px increment")
                
                # Show polar transform parameters
                if all_params['polar']['enable_polar']:
                    print(f"  • Polar Transform: Enabled ({all_params['polar']['polar_angles']}° resolution)")
                    if all_params['polar']['center_offset_x'] != 0 or all_params['polar']['center_offset_y'] != 0:
                        print(f"    Center offset: ({all_params['polar']['center_offset_x']}, {all_params['polar']['center_offset_y']})")
                else:
                    print(f"  • Polar Transform: Disabled")
                
                # Show GLCM parameters
                if all_params['glcm']['enable_glcm']:
                    distances_str = ', '.join(map(str, all_params['glcm']['distances']))
                    angles_str = ', '.join(all_params['glcm']['angles'])
                    print(f"  • GLCM Analysis: Enabled ({all_params['glcm']['levels']} gray levels)")
                    print(f"    Distances: [{distances_str}], Angles: [{angles_str}]")
                    print(f"    Properties: All 5 properties (contrast, dissimilarity, homogeneity, energy, correlation)")
                else:
                    print(f"  • GLCM Analysis: Disabled")
                
                # Show entropy parameters
                entropy_base_text = f"base {all_params['entropy']['entropy_base']}" if all_params['entropy']['use_custom_base'] else "natural log (base e)"
                print(f"  • Entropy: {all_params['entropy']['histogram_bins']} bins, {entropy_base_text}")
                
                # Show weights summary
                total_weight = (all_params['weight']['entropy_weight'] + all_params['weight']['std_weight'] + 
                              all_params['weight']['extrema_weight'] + all_params['weight']['polar_concentric_weight'] + 
                              all_params['weight']['polar_radial_weight'] + all_params['weight']['glcm_weight'])
                print(f"  • Feature Weights (total={total_weight:.3f}): E={all_params['weight']['entropy_weight']:.3f}, S={all_params['weight']['std_weight']:.3f}, X={all_params['weight']['extrema_weight']:.3f}, PC={all_params['weight']['polar_concentric_weight']:.3f}, PR={all_params['weight']['polar_radial_weight']:.3f}, GLCM={all_params['weight']['glcm_weight']:.3f}")
                
                if abs(total_weight - 1.0) > 0.01:
                    print(f"  ⚠️  Warning: Feature weights sum to {total_weight:.3f} (should be 1.0)")
                
                # Process each file
                print(f"\n🔄 Processing {len(loaded_data)} files...")
            all_results = {}
            
            for i, (file_key, file_data) in enumerate(loaded_data.items(), 1):
                if verbose:
                    print(f"\n  📊 Processing {file_key} ({i}/{len(loaded_data)})...")
                    print(f"    Format: {file_data['format']}")
                    print(f"    Original shape: {file_data['original_shape']}")
                
                try:
                    results = optimize_PL_image_interactive(
                        file_data['image'], 
                        all_params, 
                        source_format=file_data['format']
                    )
                    
                    all_results[file_key] = results
                    
                    # Show detailed results including polar and GLCM metrics
                    if verbose:
                        print(f"    ✅ Final Score: {results['final_score']:.4f}")
                        if results.get('polar_concentric') is not None:
                            print(f"    🌀 Polar Metrics: Concentric={results['polar_concentric']:.2f}, Radial={results['polar_radial']:.2f}")
                        if results.get('glcm_score') is not None:
                            print(f"    🔬 GLCM Texture Score: {results['glcm_score']:.3f}")
                            if results.get('glcm_features'):
                                features_str = ', '.join([f'{k}={v:.2f}' for k, v in results['glcm_features'].items()])
                                print(f"    🔬 GLCM Features: {features_str}")
                    
                except Exception as e:
                    print(f"    ❌ Error processing {file_key}: {str(e)}")
                    continue
            
            if not all_results:
                print("❌ No files were successfully processed!")
                return
            
            # Generate visualizations
            if verbose:
                print(f"\n📊 Generating visualizations...")
            
            # Processing pipeline plot
            pipeline_fig = plot_processing_pipeline(all_results, all_params['crop'], all_params['radial'], all_params['polar'], all_params['norm'])
            if pipeline_fig:
                plt.show(pipeline_fig)
            
            # Comparison dashboard
            comparison_fig = plot_comparison_dashboard(all_results, loaded_data, all_params['norm'])
            if comparison_fig:
                plt.show(comparison_fig)
            
            if verbose:
                print(f"\n🎉 Analysis complete! Processed {len(all_results)} files successfully.")
            else:
                # Show minimal completion message when verbose is off
                print(f"✅ Analysis complete! {len(all_results)} files processed.")
            
            # Show polar transform summary if enabled
            if verbose and all_params['polar']['enable_polar'] and any(results.get('polar_concentric') is not None for results in all_results.values()):
                print(f"\n🌀 Polar Transform Summary:")
                for file_key, results in all_results.items():
                    if results.get('polar_concentric') is not None:
                        print(f"  • {file_key}: Concentric={results['polar_concentric']:.2f}, Radial={results['polar_radial']:.2f}")
            
        except Exception as e:
            print(f"❌ Analysis failed: {str(e)}")
            import traceback
            traceback.print_exc()

def reset_parameters(button):
    """Reset all parameters to default values"""
    with output_area:
        print("🔄 Resetting all parameters to defaults...")
        
        # Reset crop parameters
        crop_widgets['y_start'].value = 300
        crop_widgets['y_end'].value = 2450
        crop_widgets['x_start'].value = 1000
        crop_widgets['x_end'].value = 3100
        
        # Reset Bayer parameters
        rgb_handling_widgets['bayer_pattern'].value = cv2.COLOR_BAYER_RG2RGB
        
        # Reset radial parameters
        radial_widgets['initial_radius'].value = 50
        radial_widgets['radius_increment'].value = 50
        radial_widgets['extrema_threshold'].value = 0.02
        
        # Reset polar parameters
        polar_widgets['enable_polar'].value = True
        polar_widgets['polar_angles'].value = 360
        polar_widgets['center_offset_x'].value = 0
        polar_widgets['center_offset_y'].value = 0
        
        # Reset GLCM parameters
        glcm_widgets['enable_glcm'].value = True
        glcm_widgets['distances'].value = [1, 2]
        glcm_widgets['angles'].value = ['0°', '90°']
        glcm_widgets['levels'].value = 256
        
        # Reset GLCM property weights (equal weighting)
        glcm_widgets['contrast_weight'].value = 1.0
        glcm_widgets['dissimilarity_weight'].value = 1.0
        glcm_widgets['homogeneity_weight'].value = 1.0
        glcm_widgets['energy_weight'].value = 1.0
        glcm_widgets['correlation_weight'].value = 1.0
        
        # Reset entropy parameters
        entropy_widgets['histogram_bins'].value = 101
        entropy_widgets['use_custom_base'].value = False
        entropy_widgets['entropy_base'].value = 2
        
        # Reset normalization parameters
        norm_widgets['entropy_norm'].value = 1.0
        norm_widgets['std_norm'].value = 0.295093297958374
        norm_widgets['extrema_norm'].value = 0.25685287332947604
        norm_widgets['polar_concentric_norm'].value = 1000.0
        norm_widgets['polar_radial_norm'].value = 1000.0
        norm_widgets['glcm_norm'].value = 1.0
        
        # Reset weight parameters (equal weighting)
        weight_widgets['entropy_weight'].value = 1.0
        weight_widgets['std_weight'].value = 1.0
        weight_widgets['extrema_weight'].value = 1.0
        weight_widgets['polar_concentric_weight'].value = 1.0
        weight_widgets['polar_radial_weight'].value = 1.0
        weight_widgets['glcm_weight'].value = 1.0
        
        # Reset verbose output
        file_widgets['verbose'].value = True
        
        print("✅ All parameters reset to defaults!")
        print("🌀 Polar transform is now enabled by default with balanced weights")
        print("🔬 GLCM texture analysis is now enabled by default")
        print("🎛️ All weights reset to 1.0 (equal weighting with weighted averaging)")
        print("📊 Weight range: 0.0-5.0 for flexible feature importance tuning")
        print("📋 Verbose output enabled by default")

# Connect button callbacks (clear existing handlers first to prevent duplicates)
analyze_button._click_handlers.callbacks.clear()
reset_button._click_handlers.callbacks.clear()

analyze_button.on_click(run_analysis)
reset_button.on_click(reset_parameters)

print("✅ Analysis and reset callbacks configured!")

✅ Analysis and reset callbacks configured!


## 8. Example Usage & Troubleshooting

### Example File Paths Format:
```
# Windows paths example:
C:\Users\username\data\sample1.h5
C:\Users\username\data\sample2.h5

# Linux/Mac paths example:
/home/username/data/sample1.h5
/home/username/data/sample2.h5
```

### Understanding the Visualizations:

1. **Processing Pipeline Plot**: Shows each step of the analysis
   - Original image with crop region (red rectangle)
   - Cropped and standardized images
   - Radial analysis rings overlaid on the image
   - **NEW: Polar Transform visualization** (when enabled) showing the image in polar coordinates
   - **NEW: Normalized Histogram** showing the data used for entropy calculation
   - Intensity profile with detected extrema (red dots)
   - Feature scores bar chart (now includes polar metrics)

2. **Enhanced Comparison Dashboard**: Compares multiple files with additional insights
   - Feature score comparison including polar transform metrics
   - Final scores comparison
   - **NEW: Polar Transform Metrics bar chart** comparing concentric vs radial variance
   - **NEW: Histogram overlay comparison** showing intensity distributions
   - Radial intensity profiles overlay
   - **NEW: Entropy vs Final Score scatter plot**
   - **NEW: Polar Metrics Correlation plot** showing relationship between concentric and radial patterns
   - Enhanced statistics with polar transform information

### Parameter Tips:

- **Bayer Conversion**: Enable for raw sensor data that needs demosaicing
- **Crop Parameters**: Adjust to focus on the sample region, avoiding holders/backgrounds
- **Initial Radius**: Start point for radial analysis (typically 10-100 pixels)
- **Radius Increment**: Spacing between rings (affects resolution of radial analysis)  
- **Extrema Threshold**: Sensitivity for detecting peaks/valleys (0.001-0.1 range)
- **🌀 Polar Transform**: 
  - **Enable**: Check to activate polar coordinate analysis
  - **Angular Resolution**: 180-720 degrees (360° recommended, higher = more detail)
  - **Center Offset**: Fine-tune center point (±100 pixels from image center)
  - **Concentric Variance**: Higher values indicate more ring-like patterns
  - **Radial Variance**: Higher values indicate more spoke-like patterns
- **🔬 GLCM Texture Analysis**:
  - **Enable**: Check to activate Gray Level Co-occurrence Matrix analysis
  - **Distances**: Pixel separations (1-5) - multiple values capture different scales
  - **Angles**: Directional orientations (0°, 45°, 90°, 135°) for texture analysis
  - **Gray Levels**: Resolution (8-256) - higher values = more detail but slower
  - **Properties**: Texture measures (contrast, homogeneity, energy, dissimilarity, correlation)
- **Entropy Base**: Choose custom base (2 for bits) or natural log (unchecked)
- **Weights**: Should sum to 1.0 for proper normalization (now includes 6 features: entropy, STD, extrema, polar concentric, polar radial, GLCM)
- **Normalization**: Adjust based on typical ranges in your data (polar ~1000, GLCM ~1.0)

### New Polar Transform Features:

**What it measures:**
- **Concentric Patterns**: Ring-like structures, defects, or gradients from center outward
- **Radial Patterns**: Spoke-like structures, cracks, or directional features

**How it works:**
1. Transforms Cartesian image to polar coordinates (radius vs angle)
2. Measures variance along radius direction (concentric patterns)
3. Measures variance along angle direction (radial patterns)
4. Both metrics contribute to final heterogeneity score

**Interpretation:**
- High concentric variance = significant ring patterns or radial gradients
- High radial variance = directional features, cracks, or asymmetric patterns
- Both high = complex mixed patterns
- Both low = uniform or simple patterns

### New GLCM (Gray Level Co-occurrence Matrix) Features:

**What it measures:**
- **Texture Properties**: Spatial relationships between pixel intensities at specified distances and angles
- **Contrast**: Local variations and differences between neighboring pixels  
- **Homogeneity**: Uniformity of texture (inverse of contrast)
- **Energy**: Angular second moment - measures orderliness/repetition
- **Dissimilarity**: Average difference between pixel pairs
- **Correlation**: Linear dependency between pixel pairs

**How it works:**
1. Creates co-occurrence matrices showing pixel pair frequencies at specified distances/angles
2. Calculates texture properties from these matrices
3. Combines properties into composite texture score
4. Higher composite scores indicate more heterogeneous/textured regions

**Interpretation:**
- High contrast/dissimilarity = more texture variation and heterogeneity
- Low homogeneity/energy = less uniform, more complex textures
- Multiple distances capture texture at different scales
- Multiple angles capture directional texture patterns

### Troubleshooting:

- **"File not found"**: Check file paths are correct and files exist
- **"No photo found"**: H5 file doesn't contain expected image data structure
- **Bayer conversion failed**: Ensure input is 2D grayscale for Bayer conversion
- **Polar transform failed**: Check center offsets don't move center outside image bounds
- **GLCM analysis failed**: Check image has sufficient gray levels or reduce levels parameter
- **Poor polar results**: Try adjusting center offset or angular resolution
- **Poor GLCM results**: Try different distances/angles or adjust gray levels
- **Weight sum warning**: Ensure all 6 feature weights sum to 1.0
- **Memory issues**: Disable polar transform/GLCM or reduce angular resolution/gray levels
- **Widget not responding**: Re-run the cell containing the widget definitions

### Advanced Usage:

- **Comparing polar patterns**: Look for samples with high concentric variance (ring defects) vs high radial variance (directional defects)
- **Center optimization**: Use small center offsets to account for off-center samples
- **Resolution tuning**: Higher angular resolution (>360°) for detailed analysis, lower (<360°) for speed
- **GLCM texture analysis**: 
  - Compare contrast vs homogeneity to identify different texture types
  - Use multiple distances (1,2,3) to capture both fine and coarse textures
  - Select appropriate angles based on expected texture orientations
  - Adjust gray levels based on image dynamic range and computation time
- **Feature balancing**: Adjust weights based on importance of texture analysis vs other heterogeneity measures
- **Weight balancing**: Adjust polar weights based on importance of pattern analysis vs traditional metrics

## 🎉 GLCM Integration Complete!

### What's New:

**🔬 Gray Level Co-occurrence Matrix (GLCM) Analysis** has been successfully added to your heterogeneity analysis pipeline!

### New Features:

1. **GLCM Texture Analysis Tab**: 
   - Enable/disable GLCM analysis
   - Configure distances (1-5 pixels) for multi-scale texture analysis
   - Select angles (0°, 45°, 90°, 135°) for directional texture patterns
   - Adjust gray levels (8-256) for computation detail/speed balance
   - Choose texture properties: contrast, dissimilarity, homogeneity, energy, correlation

2. **Enhanced Score Calculation**:
   - Final heterogeneity score now includes 6 features (was 5)
   - New weight parameter for GLCM texture contribution
   - Weights automatically rebalanced to sum to 1.0 (≈0.167 each by default)

3. **Improved Analysis Output**:
   - GLCM composite texture score reported for each file
   - Individual GLCM properties shown (contrast, homogeneity, energy, etc.)
   - Updated parameter summaries include GLCM settings

4. **Enhanced Documentation**:
   - Updated parameter guides with GLCM explanations
   - Added troubleshooting for GLCM-specific issues
   - Advanced usage tips for texture analysis

### GLCM Texture Properties Explained:

- **Contrast**: Measures local variations - higher values = more texture heterogeneity
- **Homogeneity**: Measures uniformity - lower values = more heterogeneity (inverted in composite score)  
- **Energy**: Measures orderliness - lower values = more complex textures (inverted in composite score)
- **Dissimilarity**: Measures variation between pixel pairs - higher values = more heterogeneity
- **Correlation**: Measures linear dependencies - absolute values used in composite score

### Ready to Use:

✅ **GLCM analysis is enabled by default** with balanced settings:
- Distances: [1, 2] pixels (fine and medium scale textures)
- Angles: [0°, 90°] (horizontal and vertical patterns)  
- Gray levels: 256 (full resolution)
- Properties: [contrast, homogeneity, energy] (core texture measures)

The interactive dashboard above now includes the new GLCM tab. You can adjust all parameters and see how texture analysis contributes to your heterogeneity scores!